# Cuaderno de la sesión 1. Ver el vector antes de guardarlo

En este cuaderno no hay base de datos. El objetivo es obtener un vector y mirarlo.

Un modelo de embeddings ya entrenado convierte una frase en una lista de números. Esa lista es el vector: la posición de la frase en un mapa de muchas dimensiones. No es un resumen y no es una traducción. No se lee número a número.

El modelo de esta sesión es gratuito y entiende español (`paraphrase-multilingual-MiniLM-L12-v2`). Devuelve **384** números porque así se construyó. En DocIA+ el modelo del proyecto será Amazon Titan y devolverá **1024**. Son dos mapas distintos. No se mezclan.

Las frases son de ejercicio y ya son cortas: cada una es el trozo. No son documentos oficiales del IES.

La primera ejecución descarga el modelo, unos 450 MB. Puede tardar un par de minutos. No lo estamos entrenando; lo estamos usando.


## Instalar

Python no trae un modelo de embeddings. Esta línea descarga la librería que sabe cargar uno.

`sentence-transformers` es el programa. El modelo, `paraphrase-multilingual-MiniLM-L12-v2`, se bajará en la celda siguiente, la primera vez que se use. Sin esta librería no hay función que convierta texto en números.


In [ ]:
!pip install sentence-transformers -q
print("Librería instalada")


## Convertir una frase en una lista de números

`SentenceTransformer(...)` carga el modelo en memoria. `encode` es el embedding: entra la frase y sale la lista.

`normalize_embeddings=True` pide que la lista tenga **longitud 1**. Es lo mismo que hará Titan en DocIA+, y en el tema 3 se ve por qué conviene. Lo usaremos en todas las llamadas.

Imprimimos el texto, cuántos números tiene, su longitud y solo los diez primeros. Tienen que salir 384 números y longitud 1,0. Los diez primeros bastan para ver que son decimales, no palabras. El número 7, solo, no significa nada. El significado aparece al comparar la lista completa con otra lista del mismo modelo.

El modelo no redacta y no contesta a la frase. Solo la coloca en el mapa.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

frase = "Procedimiento de formalización de matrícula en el curso de especialización."
vector = modelo.encode(frase, normalize_embeddings=True)

print("Texto:")
print(frase)
print()
print("Cuántos números tiene el vector:", len(vector))
print("Longitud del vector:", round(float(np.linalg.norm(vector)), 4))
print("Los 10 primeros:")
print(vector[:10])


## Medir quién está cerca

La cercanía es una propiedad de los vectores. No hace falta una base de datos para verla.

Se convierten tres frases con el mismo modelo. Python resta dos listas y mide lo larga que queda esa resta. Eso es una distancia. Número más pequeño, más cerca.

Esta cuenta es la longitud de la resta entre dos listas. No es el número que imprime ChromaDB en el cuaderno siguiente. Allí, con el espacio coseno, la distancia es 1 menos el coseno. Las dos se leen igual (más pequeña, más cerca), pero tienen escalas distintas y no se comparan entre sí.

Tiene que quedar más cerca «matrícula» de «inscripción» (alrededor de 0,83) que de «cafetería» (alrededor de 1,38). Las dos primeras hablan del mismo trámite con otras palabras. La tercera no. Si saliera al revés, las tres frases no se habrían codificado con el mismo modelo.


In [ ]:
frases = {
    "matrícula": "Procedimiento de formalización de matrícula en el curso de especialización.",
    "inscripción": "Instrucciones para inscribirse en el curso.",
    "cafetería": "Menú diario de la cafetería del instituto.",
}
vectores = {nombre: modelo.encode(texto, normalize_embeddings=True) for nombre, texto in frases.items()}

cerca = np.linalg.norm(vectores["matrícula"] - vectores["inscripción"])
lejos = np.linalg.norm(vectores["matrícula"] - vectores["cafetería"])

print(f"Distancia matrícula - inscripción: {cerca:.4f}")
print(f"Distancia matrícula - cafetería:   {lejos:.4f}")
print()
if cerca < lejos:
    print("La inscripción queda más cerca de la matrícula que la cafetería.")
else:
    print("El resultado no cuadra: revisa que las tres frases se han codificado con el mismo modelo.")
